# object_contact_core.py — Notebook View

This notebook mirrors `object_contact_core.py` cell by cell.
`object_contact_core.py` is the production module imported by `object_contact_web.py`.
This file exists for reading and exploration only — do not run the full notebook top-to-bottom in one go.

## 1 — SSL Fix + Imports

In [ ]:
from __future__ import annotations

import os
import platform
import ssl
from dataclasses import dataclass
from pathlib import Path

import cv2
import numpy as np
from ultralytics import YOLO, YOLOWorld


ssl._create_default_https_context = ssl._create_unverified_context

## 2 — Configuration Dataclass

All tunable parameters live in one frozen dataclass.
`object_contact_web.py` instantiates this and passes it into `detect_contacts()`.

In [ ]:
PROJECT_ROOT = Path(__file__).resolve().parents[1]


@dataclass(frozen=True)
class ObjectContactConfig:
    camera_index: int = 0
    camera_width: int = 1280
    camera_height: int = 720
    device: str = "mps"            # "mps" Apple Silicon | "cuda" NVIDIA | "cpu"
    seg_conf: float = 0.25         # stricter: trained model, high precision
    world_conf: float = 0.20       # looser: zero-shot model, lower baseline accuracy
    iou: float = 0.45
    imgsz: int = 640
    world_skip_frames: int = 3     # run YOLOWorld every N frames to save FPS
    alert_cooldown: float = 5.0    # seconds between repeated alerts for the same item
    seg_model_path: str = str(PROJECT_ROOT / "yolov8x-seg.pt")
    world_model_path: str = "yolov8x-worldv2.pt"

## 3 — Danger Class Definitions

Two separate lists — one per model:

- **`COCO_DANGER_CLASSES`** — items in YOLOv8x-seg's training set. Produces segmentation masks → pixel-level contact detection.
- **`CUSTOM_DANGER_CLASSES`** — items not in COCO (lighters, pill bottles, etc.). Detected by YOLOWorld zero-shot via text description. Bounding boxes only → proximity-based contact detection.

Custom class names are written descriptively on purpose ("cigarette lighter" vs "lighter") because YOLOWorld matches objects by language embedding — more specific text = better recall.

In [ ]:
PERSON_CLASS = "person"

COCO_DANGER_CLASSES: set[str] = {
    "knife",
    "scissors",
    "bottle",
    "wine glass",
    "fork",
    "microwave",
    "oven",
    "toaster",
}

CUSTOM_DANGER_CLASSES: list[str] = [
    "cigarette lighter",
    "box of matches",
    "prescription medicine bottle",
    "electrical extension cord",
    "plastic grocery bag",
    "wax candle",
    "clothes iron",
    "medical syringe",
    "household cleaning spray bottle",
]

DEFAULT_MONITORED_CLASSES = set(COCO_DANGER_CLASSES) | set(CUSTOM_DANGER_CLASSES)

## 4 — Result Dataclasses

`ContactDetection` holds everything about a single detected object (person or danger item).
`FrameDetections` is the return type of `detect_contacts()` — contains all detections and the active alert list for one frame.

In [ ]:
@dataclass
class ContactDetection:
    class_name: str
    source: str                                        # "seg" | "coco" | "world"
    box: tuple[int, int, int, int]
    mask: np.ndarray | None = None
    expanded_box: tuple[int, int, int, int] | None = None
    alert: bool = False


@dataclass
class FrameDetections:
    persons: list[ContactDetection]
    dangers: list[ContactDetection]
    alerts: list[str]
    cached_world: list[tuple[str, tuple[int, int, int, int]]]

## 5 — Camera + Model Setup

In [ ]:
def play_alert_sound() -> None:
    system = platform.system()
    if system == "Darwin":
        os.system('say "Warning, danger detected" &')   # & = non-blocking
    elif system == "Windows":
        import winsound
        winsound.Beep(1000, 500)
    else:
        os.system('paplay /usr/share/sounds/alsa/Front_Left.wav &')


def open_camera(config: ObjectContactConfig) -> cv2.VideoCapture:
    # CAP_AVFOUNDATION is the macOS native backend — lower latency, better permission handling
    backend = cv2.CAP_AVFOUNDATION if platform.system() == "Darwin" else cv2.CAP_DSHOW
    cap = cv2.VideoCapture(config.camera_index, backend)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, config.camera_width)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, config.camera_height)
    return cap


def load_models(config: ObjectContactConfig) -> tuple[YOLO, YOLOWorld]:
    seg_model = YOLO(config.seg_model_path)
    world_model = YOLOWorld(config.world_model_path)
    world_model.set_classes(CUSTOM_DANGER_CLASSES)
    return seg_model, world_model

## 6 — Geometry & Mask Utilities

| Function | Purpose |
|---|---|
| `resize_mask` | Scale YOLO's internal mask (e.g. 160×160) up to frame resolution, then binarise |
| `masks_overlap` | True if any pixel is True in both masks — objects are touching |
| `expand_box` | Pad a bounding box outward by `margin` px on each side, clamped to frame bounds |
| `boxes_overlap` | Standard axis-aligned rectangle intersection test |
| `mask_intersects_box` | True if the person mask has any True pixel inside the danger box region — more efficient than `box_from_mask` + `boxes_overlap` |

In [ ]:
def resize_mask(raw_mask: np.ndarray, frame_w: int, frame_h: int) -> np.ndarray:
    return cv2.resize(raw_mask, (frame_w, frame_h)) > 0.5


def masks_overlap(mask_a: np.ndarray, mask_b: np.ndarray) -> bool:
    return bool(np.logical_and(mask_a, mask_b).any())


def expand_box(
    box: tuple[int, int, int, int],
    margin: int,
    frame_w: int,
    frame_h: int,
) -> tuple[int, int, int, int]:
    x1, y1, x2, y2 = box
    return (
        max(0, x1 - margin),
        max(0, y1 - margin),
        min(frame_w - 1, x2 + margin),
        min(frame_h - 1, y2 + margin),
    )


def boxes_overlap(a: tuple[int, int, int, int], b: tuple[int, int, int, int]) -> bool:
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    return not (ax2 < bx1 or bx2 < ax1 or ay2 < by1 or by2 < ay1)


def mask_intersects_box(mask: np.ndarray, box: tuple[int, int, int, int]) -> bool:
    x1, y1, x2, y2 = box
    return bool(mask[y1:y2 + 1, x1:x2 + 1].any())

## 7 — `detect_contacts()` — Main Per-Frame Function

Called once per frame by `object_contact_web.py`. Returns a `FrameDetections` with all persons, danger items, and active alerts.

### Contact detection priority (highest to lowest accuracy)

```
1. Both person and COCO danger item have masks  →  masks_overlap()          (pixel-level)
2. Person has mask, COCO danger item has no mask  →  mask_intersects_box()  (mask vs expanded box)
3. Neither has a mask  →  boxes_overlap(person_box, expanded_danger_box)    (bbox fallback)
4. YOLOWorld custom item (never has a mask)  →  mask_intersects_box() or boxes_overlap()
```

### YOLOWorld frame skipping

YOLOWorld runs every `world_skip_frames` frames (default: every 3rd). On skipped frames, the previous `cached_world` results are reused. Dangerous objects move slowly so a 3-frame-old detection is still valid, and FPS stays smooth.

In [ ]:
def detect_contacts(
    frame: np.ndarray,
    seg_model: YOLO,
    world_model: YOLOWorld,
    config: ObjectContactConfig,
    monitored_classes: set[str],
    proximity_px: int,
    frame_count: int,
    cached_world: list[tuple[str, tuple[int, int, int, int]]],
) -> FrameDetections:
    frame_h, frame_w = frame.shape[:2]

    # ── Primary model: YOLOv8x-seg ───────────────────────────────────────────
    seg_results = seg_model.track(
        source=frame,
        persist=True,
        conf=config.seg_conf,
        iou=config.iou,
        device=config.device,
        imgsz=config.imgsz,
        verbose=False,
    )
    r_seg = seg_results[0]

    persons: list[ContactDetection] = []
    coco_dangers: list[ContactDetection] = []

    if r_seg.boxes is not None:
        has_masks = r_seg.masks is not None
        masks_data = r_seg.masks.data.cpu().numpy() if has_masks else None

        for i, box in enumerate(r_seg.boxes):
            class_name = seg_model.names[int(box.cls[0].item())]
            xyxy = tuple(map(int, box.xyxy[0].tolist()))
            mask = None
            if has_masks and masks_data is not None and i < len(masks_data):
                mask = resize_mask(masks_data[i], frame_w, frame_h)

            if class_name == PERSON_CLASS:
                persons.append(ContactDetection(class_name, "seg", xyxy, mask=mask))
            elif class_name in COCO_DANGER_CLASSES and class_name in monitored_classes:
                coco_dangers.append(ContactDetection(class_name, "coco", xyxy, mask=mask))

    # ── Secondary model: YOLOWorld (every N frames) ───────────────────────────
    if frame_count % config.world_skip_frames == 0:
        cached_world = []
        world_results = world_model.predict(
            source=frame,
            conf=config.world_conf,
            iou=config.iou,
            device=config.device,
            imgsz=config.imgsz,
            verbose=False,
        )
        r_world = world_results[0]
        if r_world.boxes is not None:
            for box in r_world.boxes:
                class_name = world_model.names[int(box.cls[0].item())]
                if class_name not in monitored_classes:
                    continue
                xyxy = tuple(map(int, box.xyxy[0].tolist()))
                cached_world.append((class_name, xyxy))

    # ── Contact detection: COCO danger × person ───────────────────────────────
    dangers: list[ContactDetection] = []
    alerts: list[str] = []

    for danger in coco_dangers:
        expanded = expand_box(danger.box, proximity_px, frame_w, frame_h)
        danger.expanded_box = expanded

        for person in persons:
            if danger.mask is not None and person.mask is not None:
                if masks_overlap(person.mask, danger.mask):
                    danger.alert = True
                    break
            elif person.mask is not None:
                if mask_intersects_box(person.mask, expanded):
                    danger.alert = True
                    break
            elif boxes_overlap(person.box, expanded):
                danger.alert = True
                break

        if danger.alert:
            alerts.append(danger.class_name)
        dangers.append(danger)

    # ── Contact detection: YOLOWorld custom × person ──────────────────────────
    for class_name, box in cached_world:
        expanded = expand_box(box, proximity_px, frame_w, frame_h)
        danger = ContactDetection(class_name, "world", box, expanded_box=expanded)

        for person in persons:
            if person.mask is not None:
                if mask_intersects_box(person.mask, expanded):
                    danger.alert = True
                    break
            elif boxes_overlap(person.box, expanded):
                danger.alert = True
                break

        if danger.alert:
            alerts.append(class_name)
        dangers.append(danger)

    return FrameDetections(persons, dangers, list(dict.fromkeys(alerts)), cached_world)